<a href="https://colab.research.google.com/github/abhishek09827/practice-notebook/blob/main/gpt_dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/abhishek09827/practice-notebook/refs/heads/main/500_maulik_darshanik_sanwaad.txt

--2026-06-28 14:33:46--  https://raw.githubusercontent.com/abhishek09827/practice-notebook/refs/heads/main/500_maulik_darshanik_sanwaad.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 296274 (289K) [text/plain]
Saving to: ‘500_maulik_darshanik_sanwaad.txt’

500_maulik_darshani 100%[===================>] 289.33K  --.-KB/s    in 0.01s   

2026-06-28 14:33:47 (28.9 MB/s) - ‘500_maulik_darshanik_sanwaad.txt’ saved [296274/296274]



In [2]:
with open('/content/500_maulik_darshanik_sanwaad.txt', 'r', encoding="utf-8") as f:
  text = f.read()

In [3]:
print("length of file" , len(text))

length of file 118696


In [4]:
print(text[:100])

500 मौलिक दार्शनिक संवाद (ओशो-प्रेरित शैली, मूल रचना)

संवाद 1

विषय: मौन

प्रश्न: मौन क्या है? उत्त


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 (),-.0123456789:?ँंअआईउएओऔकखगचछजटडणतथदधनपबभमयरलवशषसह़ािीुूृेैोौ्।
67


In [6]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("मौलिक दार्शनिक संवाद"))
print(decode(encode("मौलिक दार्शनिक संवाद")))

[45, 64, 48, 56, 28, 1, 39, 55, 47, 65, 50, 41, 56, 28, 1, 52, 20, 49, 55, 39]
मौलिक दार्शनिक संवाद


In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([118696]) torch.int64
tensor([12,  7,  7,  1, 45, 64, 48, 56, 28,  1, 39, 55, 47, 65, 50, 41, 56, 28,
         1, 52, 20, 49, 55, 39,  1,  2, 26, 50, 63,  5, 42, 65, 47, 61, 47, 56,
        37,  1, 50, 62, 48, 57,  4,  1, 45, 59, 48,  1, 47, 31, 41, 55,  3,  0,
         0, 52, 20, 49, 55, 39,  1,  8,  0,  0, 49, 56, 51, 46, 17,  1, 45, 64,
        41,  0,  0, 42, 65, 47, 50, 65, 41, 17,  1, 45, 64, 41,  1, 28, 65, 46,
        55,  1, 53, 62, 18,  1, 24, 37, 65, 37])


In [8]:
n = int(0.9 * len(data))
train = data[:n]
val = data[n:]

In [9]:
block_size = 8
train[:block_size+1]

tensor([12,  7,  7,  1, 45, 64, 48, 56, 28])

In [10]:
x = train[:block_size]
y = train[1:block_size+1]
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([12]) the target: 7
when input is tensor([12,  7]) the target: 7
when input is tensor([12,  7,  7]) the target: 1
when input is tensor([12,  7,  7,  1]) the target: 45
when input is tensor([12,  7,  7,  1, 45]) the target: 64
when input is tensor([12,  7,  7,  1, 45, 64]) the target: 48
when input is tensor([12,  7,  7,  1, 45, 64, 48]) the target: 56
when input is tensor([12,  7,  7,  1, 45, 64, 48, 56]) the target: 28


In [11]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
   data = train if split == 'train' else val
   ix = torch.randint(len(data) - block_size, (batch_size,))
   x = torch.stack([data[i:i+block_size] for i in ix])
   y = torch.stack([data[i+1:i+block_size+1] for i in ix])
   return x, y

xb, yb = get_batch('train')
print('inputs: ')
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)

print('--------')
for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b, t]
    print(f"when input is {context.tolist()} the target: {target}")

inputs: 
torch.Size([4, 8])
tensor([[20, 49, 55, 39,  1,  9, 12, 10],
        [ 1, 53, 62,  6,  0,  0, 52, 20],
        [28, 61, 20, 39, 65, 47,  1, 53],
        [61, 45,  0,  0, 42, 65, 47, 50]])
targets: 
torch.Size([4, 8])
tensor([[49, 55, 39,  1,  9, 12, 10,  0],
        [53, 62,  6,  0,  0, 52, 20, 49],
        [61, 20, 39, 65, 47,  1, 53, 62],
        [45,  0,  0, 42, 65, 47, 50, 65]])
--------
when input is [20] the target: 49
when input is [20, 49] the target: 55
when input is [20, 49, 55] the target: 39
when input is [20, 49, 55, 39] the target: 1
when input is [20, 49, 55, 39, 1] the target: 9
when input is [20, 49, 55, 39, 1, 9] the target: 12
when input is [20, 49, 55, 39, 1, 9, 12] the target: 10
when input is [20, 49, 55, 39, 1, 9, 12, 10] the target: 0
when input is [1] the target: 53
when input is [1, 53] the target: 62
when input is [1, 53, 62] the target: 6
when input is [1, 53, 62, 6] the target: 0
when input is [1, 53, 62, 6, 0] the target: 0
when input is [1, 53, 6

In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
          return logits, None
        else:
          B, T, C = logits.shape
          logits = logits.view(B*T, C)
          targets = targets.view(B*T)
          loss = F.cross_entropy(logits, targets)
          return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

xb, yb = get_batch('train')
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

prompt = "ध्यान"

idx = torch.tensor([encode(prompt)], dtype=torch.long)
# idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 67])
tensor(4.5790, grad_fn=<NllLossBackward0>)
ध्यानईँहआ1अ।श,)खछ4गतछ3दचपज.चउआअभथृैणआ6छल(म264ट6।ाआो:ीैष।नपत5ए़ण,र3
थै7ईृ4.0गउईछ।क0आच।  -छरत3ह8ो,शागएर.च2र


In [13]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [14]:
batch_size = 32
for steps in range(1000):
  xb, yb = get_batch('train')

  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

3.458610773086548


In [15]:
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))

ध्यानेणचपपथतथदऔईटा7स8शएैौब ै  .छ।16
त8
ुआन8िथँ।ईटँतीै.़ीमण उ-्खशंल:2आ9ई)(ोनक4ूकडै
वआलहीष64ँखाआृथृ2ओथ9 ैषयह: जी
वई।वणचवँँभछ3हजथग्न।गधथणौडृध2थऔ1़ोषई(ँभछ9ूओईी
ज ौखानअजाटज.79भछ।यचवैअसणभिष7
जचिधािईभऔाग)ओ54ओा1बा ै।उण1ू30ैप9व?ज
शै्4तँसलय5 दच2आेज0जपृ.तूरड्,5 :ोै?ै पन ग6ूैििथज.शक णै?भिवत82णएत8ृ़तखाल4वूथँड8शृमणिकदूयड0ब.8्त6
वरा व्7:ंकत णो5ध1औभ जआ4-ए चाथभथगव14ै
जकलह दूयाणह17?ाव।ंईँक दौ
?ेध्र989ु2णानिष6लची
4खाौअ़एआृ्7)छ9षबा.(गृ्.81यैौछृ3
दे.ए51े.ा.अ47बाआूबद(आ6ागमै।्.
दनअमभए्एिहश:गच87
8ई)उरव?े.8.)कड17्ष5ँअु े ैबद


In [16]:
generated = m.generate(idx, max_new_tokens=10000)[0]
print(generated.tolist())
print(decode(generated.tolist()))

[40, 65, 46, 55, 41, 26, 45, 43, 25, 17, 3, 6, 31, 35, 1, 12, 54, 63, 8, 13, 58, 27, 12, 45, 2, 62, 43, 35, 17, 5, 62, 10, 0, 2, 46, 18, 62, 59, 34, 55, 24, 13, 59, 36, 64, 60, 54, 40, 58, 23, 60, 45, 44, 7, 59, 48, 1, 44, 27, 44, 15, 9, 14, 52, 55, 44, 65, 21, 42, 50, 3, 36, 21, 9, 41, 48, 4, 8, 15, 50, 20, 2, 45, 46, 24, 64, 49, 11, 66, 50, 29, 55, 1, 65, 45, 7, 3, 62, 64, 45, 13, 32, 39, 41, 56, 41, 36, 53, 57, 36, 32, 35, 42, 38, 59, 38, 21, 49, 43, 55, 39, 18, 10, 16, 48, 45, 36, 34, 51, 35, 59, 47, 24, 13, 11, 36, 46, 12, 61, 20, 1, 46, 18, 41, 61, 61, 65, 2, 22, 53, 1, 61, 50, 40, 63, 1, 45, 14, 9, 11, 36, 58, 43, 39, 46, 10, 53, 10, 34, 51, 41, 56, 61, 31, 52, 38, 23, 34, 28, 1, 33, 60, 32, 10, 14, 1, 25, 36, 11, 27, 14, 0, 58, 45, 28, 10, 54, 43, 1, 33, 12, 29, 24, 25, 54, 64, 15, 14, 17, 64, 11, 48, 35, 12, 45, 50, 40, 13, 5, 11, 36, 42, 27, 7, 22, 48, 30, 53, 17, 51, 62, 64, 15, 21, 14, 15, 51, 39, 1, 6, 0, 52, 28, 58, 37, 17, 39, 40, 13, 11, 61, 35, 39, 19, 44, 32, 31, 1, 2

In [17]:
torch.manual_seed(1337)
B,T,C = 4, 8, 2
x = torch.randn(B,T,C)
x.shape


torch.Size([4, 8, 2])

In [18]:
xbow = torch.zeros((B,T,C))
for b in range(B):
  for t in range(T):
    xprev = x[b, :t+1]
    xbow[b, t] = torch.mean(xprev, 0)

In [19]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
torch.allclose(xbow, xbow2)

False

In [20]:
xbow2.shape

torch.Size([4, 8, 2])

In [21]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [22]:
print((xbow - xbow3).abs().max())

tensor(3.2363e-08)


In [24]:
torch.manual_seed(1337)
B,T,C = 4, 8, 32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
# out = wei @ x

out.shape

torch.Size([4, 8, 16])

attention vs self attention
self attenttion cant tolerate very high learning rate --> increase epochs because learning rate is low

In [23]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
m.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            X, Y = X.to(device), Y.to(device)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out


# Existing optimizer is already defined in MxuL-Q3SWSiB, so we just continue with training

print(f"Initial loss: {estimate_loss()}")

for iter in range(5000): # Increased to 5000 training iterations for better convergence
    # every once in a while evaluate the loss on train and val sets
    if iter % 500 == 0 or iter == 4999:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Final train loss: {loss.item()}")

# Generate from the model after training
prompt = "ध्यान"
idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
print("\n--- Generated text after training ---")
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))